# Airline AI Assistant

## Importing the libraries

In [8]:
import os
import json
import base64
import requests
from io import BytesIO
from PIL import Image
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

## Loading API Keys

In [33]:
load_dotenv(override = True)
nvidia_api_key = os.getenv("NVIDIA_API_KEY")
easyvoice_api_key = os.getenv("EASYVOICE_API_KEY")

if not nvidia_api_key or not easyvoice_api_key:
    print("No API key found!")
else:
    print("API key found!")

API key found!


## Initializing Base URL

In [10]:
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"

nvidia = OpenAI(
    base_url = NVIDIA_BASE_URL,
    api_key = nvidia_api_key
)

In [11]:
EASYVOICE_BASE_URL = "https://easyvoice.ae/api/v1"

easyvoice = OpenAI(
    base_url = EASYVOICE_BASE_URL,
    api_key = easyvoice_api_key
)

## Generating text to speech

In [39]:
def tts(text):
    with easyvoice.audio.speech.with_streaming_response.create(
        model = "kokoro-82m",
        voice = "af_aoede",
        input = text
    ) as response:
        response.stream_to_file("output.mp3")
    
    return "output.mp3"

## Generating image

In [13]:
def img(city):
    invoke_url = "https://ai.api.nvidia.com/v1/genai/black-forest-labs/flux.2-klein-4b"

    headers = {
        "Authorization": f"Bearer {nvidia_api_key}",
        "Accept": "application/json",
    }

    payload = {
        "prompt": f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a realistic style",
        "width": 1024,
        "height": 1024,
        "steps": 4,
        "seed": 42
    }

    response = requests.post(
        invoke_url,
        headers = headers,
        json = payload
    )

    response.raise_for_status()
    response_body = response.json()
    image_base64 = response_body["artifacts"][0]["base64"]
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

## Defining system prompt

In [14]:
system_prompt = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

## Creating a SQL database

In [15]:
import sqlite3

In [16]:
DB = "ticket_prices.db"

In [ ]:
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)")
    conn.commit()

## Creating Tools

In [18]:
def get_ticket_prices_db(city):

    print(f"Database tool called! Getting prices for {city}", flush = True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price FROM prices WHERE city = ?", (city.lower(), ))
        result = cursor.fetchone()
        return f"The price of a ticket to {city} is ${result[0]}." if result else "No price data is available for this city"

In [19]:
ticket_prices_db_desc = {
    "name": "get_ticket_prices",
    "description": "Get the price of a return ticket to the destination city!",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city that the customer wants to travel to"
            }
        }
    },
    "required": ["city"],
    "additionalProperties": False
}

In [20]:
tools = [{
    "type": "function",
    "function": ticket_prices_db_desc
}]

## Populating Database

In [74]:
def set_ticket_prices_db(city, price):
    
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?",
            (city.lower(), price, price)
        )
        conn.commit()

In [75]:
ticket_prices_db = {
    "london": 799,
    "paris": 899,
    "tokyo": 1420,
    "sydney": 2999
}

In [76]:
for city, price in ticket_prices_db.items():
    set_ticket_prices_db(city, price)

## Chat function

In [21]:
def handle_tool_calls_db(message):

    responses = []
    cities = []
    
    for tool_call in message.tool_calls:
        
        if tool_call.function.name == "get_ticket_prices":

            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("city")
            cities.append(city)
            price_details = get_ticket_prices_db(city)

            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })

    return responses, cities

In [35]:
def chat_db(history):

    history = [{
        "role": h["role"],
        "content": h["content"]
    } for h in history ]

    messages = [{"role": "system", "content": system_prompt}] + history

    response = nvidia.chat.completions.create(  
        model = "nvidia/nemotron-3-super-120b-a12b",
        messages = messages,
        tools = tools
    )

    cities = []
    image = None

    while response.choices[0].finish_reason == "tool_calls":

        message = response.choices[0].message
        tool_response, cities = handle_tool_calls_db(message)
        messages.append(message)
        messages.extend(tool_response)
        response = nvidia.chat.completions.create(
            model = "nvidia/nemotron-3-super-120b-a12b",
            messages = messages
        )

    reply = response.choices[0].message.content
    history += [{
        "role": "assistant",
        "content": reply
    }]

    voice = tts(reply)

    if cities: 
        image = img(cities[0])
    
    return history, voice, image

In [36]:
def give_prompt(message, history):
    return "", history + [{
        "role": "user",
        "content": message
    }]

## Creating Gradio interface

In [37]:
with gr.Blocks() as interface:

    with gr.Row():
        chatbot = gr.Chatbot(height = 500)
        image_output = gr.Image(height = 500, interactive = False)
    
    with gr.Row():
        audio_output = gr.Audio(autoplay = True)

    with gr.Row():
        message = gr.Textbox(label = "Chat with out AI Assistant")

    message.submit(
        give_prompt,
        inputs = [message, chatbot],
        outputs = [message, chatbot]
    ).then(
        chat_db,
        inputs = chatbot,
        outputs = [chatbot, audio_output, image_output]
    )

In [41]:
interface.launch(
    inbrowser = True
)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Database tool called! Getting prices for Tokyo


In [43]:
interface.close()

Closing server running on port: 7861
